# DAY 2 — ESS 배터리 수명 예측 모델 개발 및 평가

Batch 1·2만 사용합니다. 초기 100사이클 피처로 제공된 `cycle_life`를 회귀 예측합니다.

실행 순서: 경로·환경 → 원본·피처 감사 → 정책 그룹 분할 → Batch 1 CV 후보 비교 → 최종 모델 고정 → Valid·Batch 2 평가 → 그래프·README → 검증.

**라벨 해석:** Batch 1의 기록 종료+1 라벨과 Batch 2의 0.88Ah 도달 라벨은 동일한 관측 방식이 아닙니다. 실제 EOL 재현 성능으로 단정하지 않습니다. 상세 내용은 `output/data_audit.json`에 저장합니다.


In [1]:
from pathlib import Path
import sys, os
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/day2-matplotlib")
start = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in (start, *start.parents) if (p/"day2/day2.md").is_file() and (p/"data").is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from IPython.display import display, Markdown, Image
from day2.src.features import load_features, save_data_audit
from day2.src.modeling import split_manifest, compare_candidates, evaluate_final
from day2.src.reporting import make_figures, export_feature_design, write_readme
from day2.src.train import verify_run, save_run_config
OUTPUT_DIR = PROJECT_ROOT/"day2/output"
(OUTPUT_DIR/"image").mkdir(parents=True, exist_ok=True)
print("Python:", sys.executable)
print("Project root:", PROJECT_ROOT)


Matplotlib is building the font cache; this may take a moment.


Python: /Users/u085/Documents/교육자료/데이터 분석/.venv/bin/python
Project root: /Users/u085/Documents/교육자료/데이터 분석


## 1. 데이터와 초기 피처 확인

원본 두 파일만 순차 처리하고 초기 100사이클 피처를 다시 계산합니다. 전체 summary는 라벨 감사·설명용 그래프에만 사용합니다. 기존 DAY 1 피처와 일치하는지 확인하며 결측 타깃은 대치하지 않습니다. opaque barcode/channel의 반복은 동일 물리 셀 판정의 근거로 사용하지 않습니다.


In [2]:
features, audit, cells_data, structures = load_features(PROJECT_ROOT)
quality, data_notes = save_data_audit(PROJECT_ROOT, features, audit, structures)
display(quality)
display(audit.loc[~audit.included, ["cell_id", "policy", "reason"]])
display(Markdown("**라벨 한계:** " + data_notes["label_limitation"]))


B1: selected fields loaded


B2: selected fields loaded


,batch,raw_cells,used_cells,first100,delta_available,labels_n_plus_1,labels_matching_crossing,observed_crossing,early_bad_QD_rows,excluded_cells
0,B1,46,46,46,46,46,0,0,2,0
1,B2,47,39,47,47,0,39,39,8,8


,cell_id,policy,reason
68,B2c22,VarCharge-2C-100cycles,missing_or_invalid_target
69,B2c23,VarCharge-4C-100cycles,missing_or_invalid_target
81,B2c35,VarCharge-6C-100cycles,missing_or_invalid_target
82,B2c36,VarCharge-8C-100cycles,missing_or_invalid_target
83,B2c37,3.6C(80%)-3.6C(SLOWCYCLE,missing_or_invalid_target
84,B2c38,4.8C(80%)-4.8C(SLOWCYCLE,missing_or_invalid_target
85,B2c39,3.6C(9%)-5C(SLOWCYCLE,missing_or_invalid_target
86,B2c40,5.2C(58%)-4C(SLOWCYCLE,missing_or_invalid_target


**라벨 한계:** B1 labels are record length + 1 with no post-cycle1 measured QD<0.88; B2 finite labels match first measured QD<0.88. Different observation/label mechanisms limit EOL interpretation.

## 2. 정책 그룹 기반 데이터 분할

기존 seed=42 분할 계획을 재사용합니다. hold-out 10셀·5정책, 학습 36셀·18정책으로 나누고 학습 부분에 GroupKFold 5-fold를 적용합니다. 학습·검증 정책이 겹치지 않는지 확인합니다. Batch 2는 후보 검색 함수에 전달하지 않습니다.


In [3]:
train, valid, test, folds, manifest = split_manifest(PROJECT_ROOT, features)
display(manifest.groupby(["batch","split"]).agg(cells=("cell_id","size"), policy_groups=("policy","nunique")))
display(manifest.loc[manifest.cv_fold.notna()].groupby("cv_fold").agg(cells=("cell_id","size"),policy_groups=("policy","nunique")))


cells  policy_groups
batch split                          
B1    Train OOF     36             18
      Valid         10              5
B2    Test          39             12

,cells,policy_groups
cv_fold,,
1,7,3
2,8,4
3,8,4
4,7,4
5,6,3


## 3. 후보·피처·파라미터 비교

중앙값, Ridge/Elastic Net, Random Forest/Extra Trees, CatBoost를 같은 CV 분할에서 비교합니다. 피처군 추가·제거, 중복 피처의 공선성 축소 유무, 타깃 로그 변환을 비교합니다. 모든 대치·피처 축소·스케일·인코딩은 학습 fold 내부에 적합합니다. 선택 기준은 CV 평균 MAPE이며 표준편차와 후보 ID로 동률을 정리합니다. 선택된 CV 점수는 검색에 사용된 점수이므로 낙관적일 수 있습니다.


In [4]:
selected, comparison = compare_candidates(PROJECT_ROOT, train, folds)
display(comparison.head(12))
print("Frozen configuration:", selected)


CV: Median delta 0.85


CV: Ridge delta 0.85


CV: Ridge delta_qd 0.85


CV:

 Ridge core 0.85


...

CV: ExtraTrees core 0.85


CV: ExtraTrees policy 0.85


CV: ExtraTrees expanded 0.85


CV: CatBoost core_category 0.85


RuntimeError: Cannot clone object CatBoostRegressor(allow_writing_files=False, cat_features=['cat__policy'], depth=2, iterations=200, l2_leaf_reg=3.0, learning_rate=0.03, loss_function='RMSE', random_seed=42, thread_count=1, verbose=False), as the constructor either does not set or modifies parameter cat_features

## 4. 최종 모델·성능·오류 분석

CV로 설정을 고정한 뒤 학습 부분 36셀로 fit합니다. 같은 최종 모델로 Valid와 Batch 2를 예측하며 Train 예측은 OOF입니다. 주지표 Train은 5개 fold MAPE 평균이고 pooled OOF MAPE는 보조값입니다. Batch 2 결과에 따라 모델을 바꾸지 않습니다. Hold-out permutation 중요도는 모델 확정 후 해석용으로만 산출합니다.


In [ ]:
model, performance, metrics, predictions = evaluate_final(PROJECT_ROOT, train, valid, test, folds, selected, comparison)
display(performance)
display(metrics)
display(predictions.query("split == 'Test'").nlargest(8, "APE_pct"))


## 5. 그래프·README 작성

그래프는 Batch 1·2만 대상으로 생성하며 기존 DAY 1 결과를 덮어쓰지 않습니다. README는 sample.md 섹션 순서를 유지하고 실제 선정 이유·성능·오류·라벨 한계를 반영합니다.


In [ ]:
figures = make_figures(PROJECT_ROOT, features, cells_data, train, comparison, predictions)
export_feature_design(PROJECT_ROOT, selected)
config = save_run_config(PROJECT_ROOT, selected, figures)
write_readme(PROJECT_ROOT, features, quality, train, valid, test, selected, comparison, performance, metrics, predictions, config)
for name in ["model_comparison.png", "actual_vs_predicted.png", "residuals.png", "test_error_groups.png", "valid_importance.png"]:
    display(Image(filename=str(OUTPUT_DIR/"image"/name)))


## 6. 재현·정합성 검증

분할 중복·OOF 완결성·저장된 성능과 예측값의 일치·학습 부분에서만 적합한 대치 통계·모델 저장 후 재로딩·소스/데이터 경로·README 내부 링크를 확인합니다. 목표 성능 달성 여부와 데이터·도메인 한계를 함께 보고합니다.


In [ ]:
verification = verify_run(PROJECT_ROOT, features, train, valid, test, folds, selected, comparison, performance, predictions, model)
display(verification)
display(Markdown(f"**최종 모델:** {selected['family']} · `{selected['feature_set']}`. 성능과 해석은 루트 README.md에 갱신했습니다. 제공 라벨의 배치 간 정의 차이와 물리 셀 독립성의 한계를 함께 해석합니다."))
